* In this assignment you will be building the **Encoder** part of the Transformer architecture.
* You will be using the **PyTorch** framework to implement the following components
  * Encoder Layer that contains
    * Multi-Head Attention (MHA) Module
    * Position-wise Feed Forward Neural Network

  * Output layer that takes the encoder output and predicts the token_ids.

  * Optionally, study whether adding positional information is helpful.
  
* **DO NOT** USE Built-in **TRANSFORMER LAYERS** as it affects the reproducibility.

* You will be given with a configuration file that contains information on various hyperparameters such as embedding dimension, vocabulary size,number heads and so on

* Use ReLU activation function and Stochastic Gradient Descent optimizer
* Here are a list of helpful Pytorch functions (does not mean you have to use all of them) for this and subsequent assignments
  * [torch.matmul](https://pytorch.org/docs/stable/generated/torch.matmul.html#torch-matmul)
  * [torch.bmm](https://pytorch.org/docs/stable/generated/torch.bmm.html)
  * torch.swapdims
  * torch.unsqueeze
  * torch.squeeze
  * torch.argmax
  * [torch.Tensor.view](https://pytorch.org/docs/stable/generated/torch.Tensor.view.html)
  * [torch.nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
  * [torch.nn.Parameter](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html)
  * torch.nn.Linear
  * torch.nn.LayerNorm
  * torch.nn.ModuleList
  * torch.nn.Sequential
  * [torch.nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
  
* Important: **Do not** set any global seeds.

* Helpful resources to get started with

 * [Annotated Transformers](https://nlp.seas.harvard.edu/annotated-transformer/)
 * [PyTorch Source code of Transformer Layer](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html)



# Import

In [1]:
import torch
from torch import Tensor

import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.nn.functional import one_hot

import torch.optim as optim

from  pprint import pprint
from yaml import safe_load
import requests
from io import BytesIO
import math

# Configuration

In [2]:
#do not edit this cell
config_url = "https://raw.githubusercontent.com/Arunprakash-A/LLM-from-scratch-PyTorch/main/config_files/enc_config.yml"
response = requests.get(config_url)
config = response.content.decode("utf-8")
config = safe_load(config)
pprint(config)

{'input': {'batch_size': 10, 'embed_dim': 32, 'seq_len': 8, 'vocab_size': 10},
 'model': {'d_ff': 128,
           'd_model': 32,
           'dk': 4,
           'dq': 4,
           'dv': 4,
           'n_heads': 8,
           'n_layers': 6}}


In [3]:
#do not edit this cell
vocab_size = config['input']['vocab_size']
batch_size = config['input']['batch_size']
seq_len = config['input']['seq_len']
embed_dim = config['input']['embed_dim']

* Here, you are directly given with the token ids
* Assume that length of all sequences are equal to the context length (T) (so that we do not need to bother about padding shorter sequences while batching)

In [4]:
# do not edit this cell
data_url = 'https://github.com/Arunprakash-A/LLM-from-scratch-PyTorch/raw/main/config_files/w1_input_tokens'
r = requests.get(data_url)
token_ids = torch.load(BytesIO(r.content))
print(token_ids)

tensor([[5, 7, 5, 6, 3, 8, 7, 5],
        [7, 2, 7, 1, 2, 1, 1, 7],
        [1, 0, 0, 3, 6, 3, 0, 8],
        [5, 0, 2, 8, 6, 5, 5, 3],
        [3, 5, 4, 8, 5, 0, 7, 3],
        [8, 6, 7, 4, 4, 4, 0, 1],
        [5, 8, 1, 0, 1, 1, 0, 3],
        [1, 7, 8, 8, 0, 5, 3, 7],
        [7, 7, 1, 4, 5, 6, 7, 0],
        [1, 7, 2, 8, 3, 0, 0, 4]])


<ipython-input-4-0fefa6fcf57f>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  token_ids = torch.load(BytesIO(r.content))


# Building the sub-layers

In [5]:
# do not edit this cell
dq = torch.tensor(config['model']['dq'])
dk = torch.tensor(config['model']['dk'])
dv = torch.tensor(config['model']['dv'])
dmodel = embed_dim
heads = torch.tensor(config['model']['n_heads'])
d_ff = config['model']['d_ff']

##Multi-Head Attention

 * Be mindful when using `torch.matmul`
 * Ensure that you understood what is being computed (because matrix product is not commutative)
 * Randomly initialize the parameters using normal distribution with the following seed values
  * $W_Q:$(seed=43)
  * $W_K:$(seed=44)
  * $W_V:$(seed=45)
  * $W_O:$(seed=46)

In [6]:
#Implementation approach 1
class MHA(nn.Module):
    def __init__(self, d_model,dq,dk,dv,num_heads):
        super(MHA, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        #assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value

        # Linear layers for transforming inputs
        torch.manual_seed(43)
        self.W_q = nn.Parameter(torch.randn(d_model, d_model)) # Query transformation
        torch.manual_seed(44)
        self.W_k = nn.Parameter(torch.randn(d_model, d_model)) # Key transformation
        torch.manual_seed(45)
        self.W_v = nn.Parameter(torch.randn(d_model, d_model)) # Value transformation
        torch.manual_seed(46)
        self.W_o = nn.Parameter(torch.randn(d_model, d_model)) # Output transformation

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(torch.matmul(Q, self.W_q))
        K = self.split_heads(torch.matmul(K, self.W_k))
        V = self.split_heads(torch.matmul(V, self.W_v))

        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)

        # Combine heads and apply output transformation
        output = torch.matmul(self.combine_heads(attn_output), self.W_o)
        return output

In [29]:
#Implementation approach 2
class MHA(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHA,self).__init__()
    # your code goes here
    self.dk = dk
    torch.manual_seed(43)
    self.W_q = nn.Parameter(torch.randn(heads, dmodel, dq))
    torch.manual_seed(44)
    self.W_k = nn.Parameter(torch.randn(heads, dmodel, dk))
    torch.manual_seed(45)
    self.W_v = nn.Parameter(torch.randn(heads, dmodel, dv))
    torch.manual_seed(46)
    self.W_o = nn.Parameter(torch.randn(dmodel, dmodel))
  # your method definitions go here (if you want to)

  def forward(self,H=None):
    '''
    Input: Size [BSxTxdmodel]
    Output: Size[BSxTxdmodel]
    '''
    # your code goes here
    batch_size, seq_length, _ = H.size()
    Q = torch.matmul(H.unsqueeze(1), self.W_q)
    K = torch.matmul(H.unsqueeze(1), self.W_k)
    V = torch.matmul(H.unsqueeze(1), self.W_v)
    attn_score = torch.matmul(Q, K.transpose(2,3))/math.sqrt(self.dk)
    attn_score = torch.softmax(attn_score, dim = -1)
    z = torch.matmul(attn_score, V)
    z = z.permute(0,2,1,3).contiguous().view(batch_size, seq_length, -1)
    out = torch.matmul(z, self.W_o)
    #print(out)
    return out

## Pointwise FFN

* Randomly initialize the parameters using normal distribution with the following seed values
  * $W_{1}:$(seed=47)
  * $W_2:$(seed=48)  

In [30]:
class FFN(nn.Module):
  def __init__(self,dmodel,d_ff,layer=0):
    super(FFN,self).__init__()
    #your code goes here
    torch.manual_seed(47)
    self.W_1 = nn.Parameter(torch.randn(dmodel, d_ff))
    #nn.init.normal_(self.W_1.weight)
    torch.manual_seed(48)
    self.W_2 = nn.Parameter(torch.randn(d_ff, dmodel))
    #nn.init.normal_(self.W_2.weight)
    self.relu = nn.ReLU()


  def forward(self,x):
    '''
    input: size [BSxTxdmodel]
    output: size [BSxTxdmodel]
    '''
    #your code goes here
    out = torch.matmul(x, self.W_1)
    out = self.relu(out)
    out = torch.matmul(out, self.W_2)
    return out

## Output Layer

* Randomly initialize the linear layer
 * $W_L:$(seed=49)


In [31]:
class OutputLayer(nn.Module):

  def __init__(self,dmodel,vocab_size):
    super(OutputLayer,self).__init__()
    # your code goes here
    torch.manual_seed(49)
    self.W_l = nn.Parameter(torch.randn(dmodel, vocab_size))
    #nn.init.normal_(self.W_l.weight)

  def forward(self,representations):
    '''
    input: size [bsxTxdmodel]
    output: size [bsxTxvocab_size]
    Note: Do not apply the softmax. Just return the output of linear transformation
    '''
    out = torch.matmul(representations, self.W_l)
    return out

## Encoder Layer

In [43]:
class EncoderLayer(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,d_ff,heads):
    super(EncoderLayer,self).__init__()
    self.mha = MHA(dmodel,dq,dk,dv,heads)
    self.layer_norm_mha = torch.nn.LayerNorm(dmodel)
    self.layer_norm_ffn = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,x):

    # do a forward pass
    out = self.mha(x)#,x,x)
    out = self.layer_norm_mha(out)
    out = self.ffn(out)
    out = self.layer_norm_ffn(out)
    return out

## Model with one encoder layer

 * The encoders' forward function accepts the token_ids as input
 * Generate the embeddings for the token ids by initializing the emebedding weights from normal distribution by setting the seed value to 50
 * Use `torch.nn.Embed()` to generate required embeddings

In [44]:
class Encoder(nn.Module):

  def __init__(self,vocab_size,embed_dim,dq,dk,dv,d_ff,heads,num_layers=1):
    super(Encoder,self).__init__()
    # your code goes here
    torch.manual_seed(50)
    self.embed_weights = nn.Parameter(torch.randn(vocab_size, embed_dim))
    self.embed = nn.Embedding(vocab_size, embed_dim, _weight=self.embed_weights)
    #self.position_enc = PositionalEncoding(embed_dim)
    self.encoder = EncoderLayer(embed_dim,dq,dk,dv,d_ff,heads)
    self.output = OutputLayer(dmodel, vocab_size)

  def forward(self,x):
    '''
    The input should be tokens ids of size [BS,T]
    '''
    out = self.embed(x) #get the embeddings of the tokens
    #out = self.position_enc(out)
    out = self.encoder(out) # pass the embeddings throught the encoder layers
    out = self.output(out)# get the logits

    return out

In [45]:
model = Encoder(vocab_size,dmodel,dq,dk,dv,d_ff,heads)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Training the model

 * Train the model for 30 epochs and compute the loss

In [46]:
def train(token_ids,epochs=None):

  for epoch in range(epochs):
    out = model(token_ids)
    out = out.view(-1, vocab_size)
    target = token_ids.view(-1)
    loss = criterion(out, target)
    print(f'Loss in Epoch {epoch+1} is {loss}')
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

train(token_ids, 30)

Loss in Epoch 1 is 10.130553245544434
Loss in Epoch 2 is 8.883288383483887
Loss in Epoch 3 is 8.085538864135742
Loss in Epoch 4 is 7.347620487213135
Loss in Epoch 5 is 6.797826290130615
Loss in Epoch 6 is 6.4101152420043945
Loss in Epoch 7 is 6.067647933959961
Loss in Epoch 8 is 5.747421741485596
Loss in Epoch 9 is 5.4384331703186035
Loss in Epoch 10 is 5.161822319030762
Loss in Epoch 11 is 4.898842811584473
Loss in Epoch 12 is 4.676114082336426
Loss in Epoch 13 is 4.449737071990967
Loss in Epoch 14 is 4.269766807556152
Loss in Epoch 15 is 4.049200534820557
Loss in Epoch 16 is 3.8772361278533936
Loss in Epoch 17 is 3.587311267852783
Loss in Epoch 18 is 3.39068865776062
Loss in Epoch 19 is 3.2567920684814453
Loss in Epoch 20 is 3.173095941543579
Loss in Epoch 21 is 3.049271583557129
Loss in Epoch 22 is 2.9856598377227783
Loss in Epoch 23 is 2.880030870437622
Loss in Epoch 24 is 2.8190646171569824
Loss in Epoch 25 is 2.725048303604126
Loss in Epoch 26 is 2.671295642852783
Loss in Epoch 2

# Inference

In [47]:
with torch.inference_mode():
  predictions = torch.argmax(model(token_ids),dim = -1) # predict the labels

* See how many labels are correctly predicted

In [48]:
print(torch.count_nonzero(token_ids==predictions))

tensor(38)


* The loss by now should be about 2.39 and the number of correct predictions should be about 37

# Encoder with N Layers

  * The intialized parameters in all layers are identical
  * use ModuleList to create **deep-copies** of encoder layer

In [ ]:
import copy

In [ ]:
class Encoder(nn.Module):

  def __init__(self,vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers=1):
    super(Encoder,self).__init__()
    torch.manual_seed(50)
    self.embed_weights = nn.Embedding(vocab_size, dmodel)
    #self.position_enc = PositionalEncoding(embed_dim)
    self.enc_layers = nn.ModuleList(copy.deepcopy(EncoderLayer(dmodel, dq,dk,dv,d_ff,heads)) for i in range(num_layers))
    self.output = OutputLayer(dmodel,vocab_size)


  def forward(self,x):
    '''
    1. Get embeddings
    2. Pass it through encoder layer-1 and recursively pass the output to subsequent enc.layers
    3. output the logits
    '''
    out = self.embed_weights(x)
    #out = self.position_enc(out)
    for enc_layer in self.enc_layers:
      out = enc_layer(out)
    out = self.output(out)
    return out

* Train the stack of encoder layers with `num_layers=2` for the same 30 epochs

In [ ]:
model = Encoder(vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
def train(token_ids,epochs=None):

  for epoch in range(epochs):
    out = model(token_ids)
    out = out.view(-1, vocab_size)
    target = token_ids.view(-1)
    loss = criterion(out, target)
    print(f'Loss in Epoch {epoch+1} is {loss}')
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()


In [ ]:
train(token_ids,30)

Loss in Epoch 1 is 11.906225204467773
Loss in Epoch 2 is 8.679384231567383
Loss in Epoch 3 is 7.405575752258301
Loss in Epoch 4 is 6.625384330749512
Loss in Epoch 5 is 6.027838706970215
Loss in Epoch 6 is 5.419787406921387
Loss in Epoch 7 is 5.039213180541992
Loss in Epoch 8 is 4.560025215148926
Loss in Epoch 9 is 4.258828639984131
Loss in Epoch 10 is 3.9945387840270996
Loss in Epoch 11 is 3.8670761585235596
Loss in Epoch 12 is 3.643892765045166
Loss in Epoch 13 is 3.3925864696502686
Loss in Epoch 14 is 3.174222469329834
Loss in Epoch 15 is 3.10710072517395
Loss in Epoch 16 is 2.8819620609283447
Loss in Epoch 17 is 2.828768253326416
Loss in Epoch 18 is 2.7548890113830566
Loss in Epoch 19 is 2.6378273963928223
Loss in Epoch 20 is 2.5472524166107178
Loss in Epoch 21 is 2.474602222442627
Loss in Epoch 22 is 2.45007586479187
Loss in Epoch 23 is 2.3400559425354004
Loss in Epoch 24 is 2.268303394317627
Loss in Epoch 25 is 2.218679666519165
Loss in Epoch 26 is 2.1799659729003906
Loss in Epoch

In [ ]:
with torch.inference_mode():
  predictions = torch.argmax(model(token_ids),dim = -1) # predict the labels

In [ ]:
torch.count_nonzero(predictions==token_ids)

tensor(37)

* Now, the loss value should be about 1.9 and the number of correct preditions is about 38

## Count Number of Parameters

In [ ]:
total_num_parameters = 0
for parameter in model.parameters():
  #print(parameter.view(-1).shape[0])
  total_num_parameters += parameter.view(-1).shape[0]

print('total number of parameters in the model \n including the embedding layer is:', total_num_parameters)

total number of parameters in the model 
 including the embedding layer is: 25472


## (Optional) Positional Encoding

 * We now use the positional embedding as defined in the [paper](https://arxiv.org/pdf/1706.03762v1.pdf) (differs a bit from the lecture).
 * Note that, the positional encoding for each position is fixed (not a learnable parameter)
 * However, we add this with the raw_embeddings which are learnable.
 * Therefore, it is important to create a class definition for PE and register PE parameters in the buffer (in case we move the model to GPU)
 * Just create a matrix of same size of input and add it to the embeddings

In [ ]:
import math
class PositionalEncoding(nn.Module):
    "Implement the PE function."

    def __init__(self,d_model,max_len=8):
        super(PositionalEncoding, self).__init__()

        #compute it in the log space
        pe = torch.arange(0, max_len*d_model).reshape(max_len, d_model).float()
        pe[0:,0::2] = torch.sin(torch.exp(torch.log(pe[0:, 0::2]%d_model)-(2*(pe[0:, 0::2]//d_model)/d_model)*torch.log(torch.tensor(10000))))
        pe[0:,1::2] = torch.cos(torch.exp(torch.log(pe[0:, 1::2]%d_model)-(2*(pe[0:, 1::2]//d_model)/d_model)*torch.log(torch.tensor(10000))))
        self.register_buffer("pe", pe)

    def forward(self, x):
        # add positional embedding
        x = x + self.pe
        return x
